# Práctica 3 — AWS Braket en Colab

**Introducción a la computación cuántica** · Centro de Excelencia en Computación Cuántica e IA
Universidad Nacional de Colombia, Sede Medellín

En esta práctica ejecutamos circuitos cuánticos con el **SDK de Amazon Braket**, primero en su **simulador local** (gratis, sin cuenta de AWS, corre dentro de Colab) y al final, de forma **opcional**, en la nube de AWS (simuladores gestionados y hardware real).

> Ejecuta las celdas en orden con **Shift+Enter**. La parte del simulador local no necesita AWS ni tarjeta: funciona para todos.

## 0. Instalación (una vez por sesión de Colab)

In [ ]:
!pip install amazon-braket-sdk -q
print('Braket SDK instalado')

## 1. Tu primer qubit (simulador local)

Ponemos un qubit en **superposición** con la compuerta de Hadamard y lo medimos 1000 veces. Deberíamos ver ~50/50.

In [ ]:
from braket.circuits import Circuit
from braket.devices import LocalSimulator

circ = Circuit().h(0)          # Hadamard sobre el qubit 0
print(circ)

device = LocalSimulator()      # simulador local: gratis, sin AWS
result = device.run(circ, shots=1000).result()
counts = result.measurement_counts
print("Conteos:", counts)

## 2. Entrelazamiento: el estado de Bell

`h(0).cnot(0,1)` crea el estado \((|00\rangle+|11\rangle)/\sqrt2\). Al medir solo salen **00** y **11** (nunca 01 ni 10): los dos qubits están correlacionados.

In [ ]:
bell = Circuit().h(0).cnot(0, 1)
print(bell)
counts_bell = LocalSimulator().run(bell, shots=1000).result().measurement_counts
print("Bell:", counts_bell)

## 3. Tres qubits: el estado GHZ

In [ ]:
ghz = Circuit().h(0).cnot(0, 1).cnot(1, 2)
counts_ghz = LocalSimulator().run(ghz, shots=1000).result().measurement_counts
print("GHZ:", counts_ghz)   # ~ mitad 000, mitad 111

## 4. Visualizar el resultado

In [ ]:
import matplotlib.pyplot as plt

def graficar(counts, titulo):
    etiquetas = sorted(counts)
    valores = [counts[k] for k in etiquetas]
    plt.figure(figsize=(5,3))
    plt.bar(etiquetas, valores, color="#6d28d9")
    plt.title(titulo); plt.ylabel("conteos"); plt.xlabel("resultado")
    plt.tight_layout(); plt.show()

graficar(counts_bell, "Estado de Bell (simulador local)")

## 5. Valor esperado sin medir (statevector)

El simulador local también puede darte el **vector de estado** exacto y valores esperados, útil para verificar la teoría.

In [ ]:
from braket.circuits import Observable

c = Circuit().h(0)
c.expectation(observable=Observable.Z(), target=0)   # <Z> para |+> debe ser 0
res = LocalSimulator().run(c, shots=0).result()       # shots=0 -> resultado exacto
print("<Z> en |+> =", res.values[0])

---

## 6. (Opcional) Ejecutar en la nube de AWS — simuladores gestionados y hardware real

⚠️ **Esta parte SÍ requiere una cuenta de AWS con Braket habilitado y TIENE COSTO** (por tarea + por shot; los QPU además tienen cola y ventanas de disponibilidad). El simulador local de arriba es gratis y suficiente para practicar.

Para usarla necesitas configurar credenciales de AWS en esta sesión de Colab. La forma más simple (para una demo tuya, no para compartir) es:

```python
import os
os.environ["AWS_ACCESS_KEY_ID"]     = "TU_ACCESS_KEY"
os.environ["AWS_SECRET_ACCESS_KEY"] = "TU_SECRET_KEY"
os.environ["AWS_DEFAULT_REGION"]    = "us-east-1"   # región del dispositivo
```

Nunca subas tus credenciales a GitHub. En un entorno real, usa credenciales temporales (STS) o roles.

In [ ]:
# Descomenta y ejecuta SOLO si configuraste credenciales de AWS.
#
# from braket.aws import AwsDevice
#
# # Simulador gestionado SV1 (en la nube, de pago por tarea):
# sv1 = AwsDevice("arn:aws:braket:::device/quantum-simulator/amazon/sv1")
# tarea = sv1.run(bell, shots=100)
# print("SV1:", tarea.result().measurement_counts)
#
# # QPU real (ejemplo; verifica el ARN vigente en la consola de Braket -> Devices):
# # qpu = AwsDevice("arn:aws:braket:us-east-1::device/qpu/ionq/Aria-1")
# # tarea = qpu.run(bell, shots=100)   # asíncrono: hay cola
# # print(tarea.result().measurement_counts)

### ¿Qué dispositivos hay disponibles? (requiere credenciales)

In [ ]:
# from braket.aws import AwsDevice
# for d in AwsDevice.get_devices():
#     print(d.name, "-", d.type, "-", d.arn)

## Para explorar (entrega)

1. Cambia la Hadamard por `x(0)` (compuerta NOT): ¿qué mide el simulador?
2. En el estado de Bell, mide 5000 shots y grafica: ¿siguen apareciendo solo 00 y 11?
3. Construye un circuito de 4 qubits tipo GHZ y comprueba que solo salen 0000 y 1111.
4. (Reto) Calcula `<Z>` en el qubit 0 del estado de Bell con `shots=0`. ¿Cuánto da y por qué?